In [ ]:
!pip install plotly kaleido -q

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Set plotly theme
pio.templates.default = "plotly_white"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.9 MB/s eta 0:00:00


In [ ]:
print("\n📁 UPLOAD YOUR ANALYSIS FILES:")
print("-" * 70)
print("Upload these files:")
print("  1. matches_cleaned.csv")
print("  2. team_performance.csv")
print("  3. venue_analysis.csv")

uploaded = files.upload()

print(f"\n✅ {len(uploaded)} files uploaded!")


📁 UPLOAD YOUR ANALYSIS FILES:
----------------------------------------------------------------------
Upload these files:
  1. matches_cleaned.csv
  2. team_performance.csv
  3. venue_analysis.csv


Saving matches_cleaned.csv to matches_cleaned.csv
Saving team_performance.csv to team_performance.csv
Saving venue_analysis.csv to venue_analysis.csv

✅ 3 files uploaded!


In [ ]:
# Load main dataset
df_matches = pd.read_csv('matches_cleaned.csv')
print(f"✅ Matches: {len(df_matches):,} loaded")

# Load analysis files
df_teams = pd.read_csv('team_performance.csv')
print(f"✅ Teams: {len(df_teams)} loaded")

df_venues = pd.read_csv('venue_analysis.csv')
print(f"✅ Venues: {len(df_venues)} loaded")

✅ Matches: 1,193 loaded
✅ Teams: 17 loaded
✅ Venues: 41 loaded


In [ ]:
# TEAM NAME STANDARDIZATION MAPPING
TEAM_MAPPING = {
    # Mumbai Indians variations
    'Mumbai Indians': 'Mumbai Indians',
    'Mumbai indians': 'Mumbai Indians',
    'Mumbai Indian': 'Mumbai Indians',
    'MI': 'Mumbai Indians',

    # Chennai Super Kings variations
    'Chennai Super Kings': 'Chennai Super Kings',
    'Chennai super kings': 'Chennai Super Kings',
    'CSK': 'Chennai Super Kings',

    # Royal Challengers Bangalore variations
    'Royal Challengers Bangalore': 'Royal Challengers Bangalore',
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Royal Challengers bangalore': 'Royal Challengers Bangalore',
    'RCB': 'Royal Challengers Bangalore',

    # Kolkata Knight Riders variations
    'Kolkata Knight Riders': 'Kolkata Knight Riders',
    'Kolkata knight riders': 'Kolkata Knight Riders',
    'KKR': 'Kolkata Knight Riders',

    # Delhi variations
    'Delhi Capitals': 'Delhi Capitals',
    'Delhi capitals': 'Delhi Capitals',
    'Delhi Daredevils': 'Delhi Capitals',
    'DC': 'Delhi Capitals',
    'DD': 'Delhi Capitals',

    # Rajasthan Royals variations
    'Rajasthan Royals': 'Rajasthan Royals',
    'Rajasthan royals': 'Rajasthan Royals',
    'RR': 'Rajasthan Royals',

    # Sunrisers Hyderabad variations
    'Sunrisers Hyderabad': 'Sunrisers Hyderabad',
    'Sunrisers hyderabad': 'Sunrisers Hyderabad',
    'SRH': 'Sunrisers Hyderabad',
    'Deccan Chargers': 'Sunrisers Hyderabad',

    # Punjab variations
    'Punjab Kings': 'Punjab Kings',
    'Kings XI Punjab': 'Punjab Kings',
    'KXIP': 'Punjab Kings',
    'PBKS': 'Punjab Kings',

    # Gujarat Titans
    'Gujarat Titans': 'Gujarat Titans',
    'GT': 'Gujarat Titans',

    # Lucknow Super Giants
    'Lucknow Super Giants': 'Lucknow Super Giants',
    'LSG': 'Lucknow Super Giants',

    # Rising Pune variations
    'Rising Pune Supergiants': 'Rising Pune Supergiants',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'RPS': 'Rising Pune Supergiants',

    # Pune Warriors
    'Pune Warriors': 'Pune Warriors',
    'Pune Warriors India': 'Pune Warriors',

    # Kochi Tuskers
    'Kochi Tuskers Kerala': 'Kochi Tuskers Kerala',
}

# VENUE NAME STANDARDIZATION MAPPING
VENUE_MAPPING = {
    # Bangalore venues
    'M Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'M. Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'Chinnaswamy Stadium': 'M Chinnaswamy Stadium',

    # Mumbai venues
    'Wankhede Stadium': 'Wankhede Stadium',
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',
    'Brabourne Stadium': 'Brabourne Stadium',
    'Brabourne Stadium, Mumbai': 'Brabourne Stadium',
    'Dr DY Patil Sports Academy': 'Dr DY Patil Sports Academy',

    # Delhi venues
    'Arun Jaitley Stadium': 'Arun Jaitley Stadium',
    'Arun Jaitley Stadium, Delhi': 'Arun Jaitley Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',

    # Chennai venues
    'MA Chidambaram Stadium': 'MA Chidambaram Stadium',
    'M A Chidambaram Stadium': 'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium',
    'Chepauk': 'MA Chidambaram Stadium',

    # Kolkata venues
    'Eden Gardens': 'Eden Gardens',
    'Eden Gardens, Kolkata': 'Eden Gardens',

    # Hyderabad venues
    'Rajiv Gandhi International Stadium': 'Rajiv Gandhi International Stadium',
    'Rajiv Gandhi Intl. Cricket Stadium': 'Rajiv Gandhi International Stadium',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium',

    # Jaipur venues
    'Sawai Mansingh Stadium': 'Sawai Mansingh Stadium',
    'Sawai Mansingh Stadium, Jaipur': 'Sawai Mansingh Stadium',

    # Mohali venues
    'Punjab Cricket Association Stadium': 'Punjab Cricket Association Stadium',
    'Punjab Cricket Association IS Bindra Stadium': 'Punjab Cricket Association Stadium',
    'IS Bindra Stadium': 'Punjab Cricket Association Stadium',

    # Ahmedabad venues
    'Narendra Modi Stadium': 'Narendra Modi Stadium',
    'Narendra Modi Stadium, Ahmedabad': 'Narendra Modi Stadium',

    # Pune venues
    'Maharashtra Cricket Association Stadium': 'Maharashtra Cricket Association Stadium',
    'MCA Stadium': 'Maharashtra Cricket Association Stadium',
    'MCA Stadium, Pune': 'Maharashtra Cricket Association Stadium',
}

In [ ]:
def clean_name(name, mapping):

    if pd.isna(name):
        return name

    # Convert to string and strip
    name = str(name).strip()

    # Check if in mapping
    if name in mapping:
        return mapping[name]

    # If not in mapping, return as-is (cleaned)
    return name
# Clean all team columns
team_columns = ['team1', 'team2', 'toss_winner', 'match_won_by']
for col in team_columns:
    if col in df_matches.columns:
        df_matches[col] = df_matches[col].apply(lambda x: clean_name(x, TEAM_MAPPING))

# Clean venue columns
if 'venue' in df_matches.columns:
    df_matches['venue'] = df_matches['venue'].apply(lambda x: clean_name(x, VENUE_MAPPING))

if 'city' in df_matches.columns:
    df_matches['city'] = df_matches['city'].apply(lambda x: str(x).strip() if pd.notna(x) else x)

print("✅ Matches data cleaned")

# Show unique teams after cleaning
print(f"\n📊 Unique teams after cleaning: {df_matches['team1'].nunique()}")
print("Teams:")
for team in sorted(df_matches['team1'].unique()):
    if pd.notna(team):
        print(f"  • {team}")


print("\n  Cleaning team performance data...")

if 'Team' in df_teams.columns:
    df_teams['Team'] = df_teams['Team'].apply(lambda x: clean_name(x, TEAM_MAPPING))

# Remove duplicates by aggregating
df_teams_clean = df_teams.groupby('Team').agg({
    'Matches': 'sum',
    'Wins': 'sum',
    'Losses': 'sum',
    'Win_Percentage': 'mean'  # Recalculate
}).reset_index()

# Recalculate win percentage
df_teams_clean['Win_Percentage'] = (
    df_teams_clean['Wins'] / df_teams_clean['Matches'] * 100
).round(2)

# Sort by win percentage
df_teams_clean = df_teams_clean.sort_values('Win_Percentage', ascending=False)

print(f"✅ Team performance cleaned")
print(f"   Teams before: {len(df_teams)}")
print(f"   Teams after: {len(df_teams_clean)}")


print("\n  Cleaning venue analysis data...")

if 'Venue' in df_venues.columns:
    df_venues['Venue'] = df_venues['Venue'].apply(lambda x: clean_name(x, VENUE_MAPPING))

# Remove duplicates by aggregating
df_venues_clean = df_venues.groupby('Venue').agg({
    'Matches': 'sum',
    'Avg_Total_Runs': 'mean',
    'Avg_1st_Innings': 'mean',
    'Avg_2nd_Innings': 'mean',
    'High_Scoring_Pct': 'mean',
    'Close_Match_Pct': 'mean'
}).reset_index()

# Round values
for col in df_venues_clean.columns:
    if col != 'Venue' and col != 'Matches':
        df_venues_clean[col] = df_venues_clean[col].round(1)

# Sort by avg total runs
df_venues_clean = df_venues_clean.sort_values('Avg_Total_Runs', ascending=False)

print(f"✅ Venue analysis cleaned")
print(f"   Venues before: {len(df_venues)}")
print(f"   Venues after: {len(df_venues_clean)}")

✅ Matches data cleaned

📊 Unique teams after cleaning: 14
Teams:
  • Chennai Super Kings
  • Delhi Capitals
  • Gujarat Lions
  • Gujarat Titans
  • Kochi Tuskers Kerala
  • Kolkata Knight Riders
  • Lucknow Super Giants
  • Mumbai Indians
  • Pune Warriors
  • Punjab Kings
  • Rajasthan Royals
  • Rising Pune Supergiants
  • Royal Challengers Bangalore
  • Sunrisers Hyderabad

  Cleaning team performance data...
✅ Team performance cleaned
   Teams before: 17
   Teams after: 14

  Cleaning venue analysis data...
✅ Venue analysis cleaned
   Venues before: 41
   Venues after: 40


### **STEP 1 : KPI Creation**

In [ ]:
# KPI 1: Total Matches
total_matches = len(df_matches)

# KPI 2: Total Teams
total_teams = len(df_teams)

# KPI 3: Average Runs
avg_runs = df_matches['total_runs'].mean()

# KPI 4: Close Matches
close_match_pct = (df_matches['close_match'].sum() / total_matches * 100)

# KPI 5: Toss Impact
toss_impact = (df_matches['toss_match_win'].sum() / total_matches * 100)

# KPI 6: Top Team
top_team = df_teams.iloc[0]['Team']
top_team_winrate = df_teams.iloc[0]['Win_Percentage']

print(f"📊 Total Matches: {total_matches}")
print(f"🏆 Teams Analyzed: {total_teams}")
print(f"⚡ Avg Runs/Match: {avg_runs:.0f}")
print(f"🎯 Close Matches: {close_match_pct:.1f}%")
print(f"🪙 Toss Impact: {toss_impact:.1f}%")
print(f"👑 Best Team: {top_team} ({top_team_winrate}%)")

📊 Total Matches: 1193
🏆 Teams Analyzed: 17
⚡ Avg Runs/Match: 321
🎯 Close Matches: 14.6%
🪙 Toss Impact: 50.8%
👑 Best Team: Gujarat Titans (60.94%)


### **STEP 2 : Dashboard Creation**

In [ ]:
# Create figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Team Win Percentage (Top 10)',
        'Runs Scored vs Conceded',
        'Top 10 Highest Scoring Venues',
        'Scoring Trend Over Seasons',
        'Toss Impact on Match Outcome',
        'Close Matches by Season'
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "scatter"}],
        [{"type": "pie"}, {"type": "bar"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.15,
    row_heights=[0.33, 0.33, 0.33]
)
# ═════════════════════════════════════════
# CHART 1: Team Win Percentage
# ═════════════════════════════════════════

top_10_teams = df_teams.head(10).sort_values('Win_Percentage', ascending=True)

fig.add_trace(
    go.Bar(
        y=top_10_teams['Team'],
        x=top_10_teams['Win_Percentage'],
        orientation='h',
        marker=dict(
            color=top_10_teams['Win_Percentage'],
            colorscale='Blues',
            showscale=False
        ),
        text=top_10_teams['Win_Percentage'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Win Rate: %{x:.1f}%<extra></extra>'
    ),
    row=1, col=1
)

# ═════════════════════════════════════════
# CHART 2: Runs Scored vs Conceded
# ═════════════════════════════════════════

# Need to load scoring data from Day 3
# For now, use team performance data
top_scoring = df_teams.head(10)

fig.add_trace(
    go.Bar(
        name='Average Performance',
        y=top_scoring['Team'],
        x=top_scoring['Wins'],
        orientation='h',
        marker_color='lightgreen',
        hovertemplate='<b>%{y}</b><br>Wins: %{x}<extra></extra>'
    ),
    row=1, col=2
)

# ═════════════════════════════════════════
# CHART 3: Top Venues
# ═════════════════════════════════════════

top_venues = df_venues.head(10).sort_values('Avg_Total_Runs', ascending=True)

fig.add_trace(
    go.Bar(
        y=top_venues['Venue'],
        x=top_venues['Avg_Total_Runs'],
        orientation='h',
        marker=dict(
            color=top_venues['Avg_Total_Runs'],
            colorscale='Oranges',
            showscale=False
        ),
        text=top_venues['Avg_Total_Runs'].apply(lambda x: f'{x:.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Avg Runs: %{x:.0f}<extra></extra>'
    ),
    row=2, col=1
)

# ═════════════════════════════════════════
# CHART 4: Scoring Trend
# ═════════════════════════════════════════

seasonal = df_matches.groupby('season')['total_runs'].mean().reset_index()
seasonal.columns = ['Season', 'Avg_Runs']

fig.add_trace(
    go.Scatter(
        x=seasonal['Season'],
        y=seasonal['Avg_Runs'],
        mode='lines+markers',
        line=dict(color='purple', width=3),
        marker=dict(size=10, color='purple'),
        hovertemplate='<b>Season %{x}</b><br>Avg Runs: %{y:.0f}<extra></extra>'
    ),
    row=2, col=2
)

# ═════════════════════════════════════════
# CHART 5: Toss Impact
# ═════════════════════════════════════════

toss_counts = df_matches['toss_match_win'].value_counts()
toss_labels = ['Toss Winner Lost', 'Toss Winner Won']

fig.add_trace(
    go.Pie(
        labels=toss_labels,
        values=toss_counts.values,
        marker=dict(colors=['#ff9999', '#90ee90']),
        hole=0.4,
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Matches: %{value}<br>Percentage: %{percent}<extra></extra>'
    ),
    row=3, col=1
)

# ═════════════════════════════════════════
# CHART 6: Close Matches by Season
# ═════════════════════════════════════════

close_by_season = df_matches.groupby('season')['close_match'].sum().reset_index()
close_by_season.columns = ['Season', 'Close_Matches']

fig.add_trace(
    go.Bar(
        x=close_by_season['Season'],
        y=close_by_season['Close_Matches'],
        marker_color='teal',
        hovertemplate='<b>Season %{x}</b><br>Close Matches: %{y}<extra></extra>'
    ),
    row=3, col=2
)

# Update layout
fig.update_layout(
    title={
        'text': f'<b>IPL INSIGHTS DASHBOARD</b><br><sub>Analyzing {total_matches} matches across {total_teams} teams</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24}
    },
    height=1400,
    showlegend=False,
    font=dict(family="Arial, sans-serif", size=11),
    plot_bgcolor='white',
    paper_bgcolor='#f8f9fa'
)

# Update all axes
fig.update_xaxes(showgrid=True, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

# Show dashboard
fig.show()


### **STEP 3 : KPI Card Creation**

In [ ]:
# Create KPI cards figure
kpi_fig = go.Figure()

# Add KPI cards as annotations
kpi_fig.add_trace(go.Scatter(
    x=[1, 2, 3, 4, 5],
    y=[1, 1, 1, 1, 1],
    mode='markers',
    marker=dict(size=1, color='white'),
    showlegend=False,
    hoverinfo='skip'
))

# KPI Cards text
kpis = [
    {'value': f'{total_matches}', 'label': 'Total Matches', 'x': 1, 'color': '#3498db'},
    {'value': f'{avg_runs:.0f}', 'label': 'Avg Runs/Match', 'x': 2, 'color': '#e74c3c'},
    {'value': f'{close_match_pct:.1f}%', 'label': 'Close Matches', 'x': 3, 'color': '#2ecc71'},
    {'value': f'{toss_impact:.1f}%', 'label': 'Toss Impact', 'x': 4, 'color': '#f39c12'},
    {'value': f'{top_team_winrate:.1f}%', 'label': f'{top_team} Win Rate', 'x': 5, 'color': '#9b59b6'}
]

for kpi in kpis:
    # Value (big number)
    kpi_fig.add_annotation(
        x=kpi['x'], y=1.1,
        text=f"<b>{kpi['value']}</b>",
        showarrow=False,
        font=dict(size=32, color=kpi['color'], family='Arial Black'),
        xanchor='center'
    )

    # Label (small text)
    kpi_fig.add_annotation(
        x=kpi['x'], y=0.9,
        text=kpi['label'],
        showarrow=False,
        font=dict(size=12, color='gray'),
        xanchor='center'
    )

kpi_fig.update_layout(
    title='<b>Key Performance Indicators</b>',
    height=250,
    xaxis=dict(visible=False, range=[0.5, 5.5]),
    yaxis=dict(visible=False, range=[0.5, 1.5]),
    plot_bgcolor='white',
    paper_bgcolor='#f8f9fa',
    margin=dict(l=20, r=20, t=60, b=20)
)

kpi_fig.show()

### **STEP 3 : Team Comparison**

In [ ]:
# Team comparison chart
team_comp_fig = px.bar(
    df_teams.head(15),
    x='Team',
    y=['Wins', 'Losses'],
    title='<b>Win-Loss Comparison (Top 15 Teams)</b>',
    labels={'value': 'Matches', 'variable': 'Result'},
    color_discrete_map={'Wins': '#2ecc71', 'Losses': '#e74c3c'},
    barmode='stack',
    height=500
)

team_comp_fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='#f8f9fa',
    font=dict(family="Arial", size=12),
    hovermode='x unified'
)

team_comp_fig.show()


### **STEP 4 : Venue Heatmap**

In [ ]:
# Top 15 venues visualization
venue_fig = go.Figure()

top_15_venues = df_venues.head(15)

# Add bars for different metrics
venue_fig.add_trace(go.Bar(
    name='Avg Total Runs',
    x=top_15_venues['Venue'],
    y=top_15_venues['Avg_Total_Runs'],
    marker_color='#3498db',
    hovertemplate='<b>%{x}</b><br>Avg Runs: %{y:.0f}<extra></extra>'
))

venue_fig.update_layout(
    title='<b>Top 15 Venues - Scoring Analysis</b>',
    xaxis_title='Venue',
    yaxis_title='Average Total Runs',
    xaxis_tickangle=-45,
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='#f8f9fa',
    font=dict(family="Arial", size=11)
)

venue_fig.show()
